In [2]:
import pandas as pd
df=pd.read_csv("Dataset/ML Intern Dataset.csv")
clean_df=df.copy()

In [3]:
clean_df=clean_df.drop_duplicates()

In [4]:
clean_df.isnull().sum()

SN                0
Train_No          0
Station_Code      0
1A                0
2A                0
3A                0
SL                0
Station_Name      0
Route_Number      0
Arrival_time      0
Departure_Time    0
Distance          0
dtype: int64

In [5]:
clean_df=clean_df.sort_values(
    ["Train_No", "Route_Number"]
).reset_index(drop=True)

In [6]:
clean_df["Arrival_Minutes"]=(
    pd.to_timedelta(clean_df["Arrival_time"]).dt.total_seconds() / 60
)

In [7]:
clean_df["Departure_Minutes"] = (
    pd.to_timedelta(clean_df["Departure_Time"]).dt.total_seconds() / 60
)

In [8]:
clean_df[
    ["Arrival_time", "Arrival_Minutes",
      "Departure_Time", "Departure_Minutes"]
].head(10)

,Arrival_time,Arrival_Minutes,Departure_Time,Departure_Minutes
0,00:00:00,0.0,10:25:00,625.0
1,11:06:00,666.0,11:08:00,668.0
2,11:28:00,688.0,11:30:00,690.0
3,12:10:00,730.0,00:00:00,0.0
4,00:00:00,0.0,20:30:00,1230.0
5,21:04:00,1264.0,21:06:00,1266.0
6,21:26:00,1286.0,21:28:00,1288.0
7,22:25:00,1345.0,00:00:00,0.0
8,19:40:00,1180.0,19:40:00,1180.0
9,20:18:00,1218.0,20:20:00,1220.0


In [9]:
first_station=clean_df.groupby("Train_No").head(1).index
last_station=clean_df.groupby("Train_No").tail(1).index

In [10]:
clean_df.loc[first_station, "Arrival_Minutes"] = pd.NA
clean_df.loc[last_station, "Departure_Minutes"] = pd.NA

In [11]:
clean_df[
    clean_df["Train_No"] == 107
][[
    "Station_Name",
    "Route_Number",
    "Arrival_time",
    "Departure_Time",
    "Arrival_Minutes",
    "Departure_Minutes",
    "Distance"
]]

,Station_Name,Route_Number,Arrival_time,Departure_Time,Arrival_Minutes,Departure_Minutes,Distance
0,SAWANTWADI R,1,00:00:00,10:25:00,NaN,625.0,0
1,THIVIM,1,11:06:00,11:08:00,666.0,668.0,32
2,KARMALI,1,11:28:00,11:30:00,688.0,690.0,49
3,MADGOAN JN.,1,12:10:00,00:00:00,730.0,NaN,78


In [12]:
clean_df= clean_df.sort_values(
    ["Train_No", "SN"]
).reset_index(drop=True)

In [13]:
def calculate_journey_duration(group):
    group = group.sort_values("SN")

    start_time = group.iloc[0]["Departure_Minutes"]
    current_time = start_time
    day_offset = 0

    for i in range(1, len(group)):

        arrival = group.iloc[i]["Arrival_Minutes"]

        if arrival < (current_time % 1440):
            day_offset += 1440

        current_time = arrival + day_offset

        if i < len(group) - 1:
            departure = group.iloc[i]["Departure_Minutes"]

            if departure < (current_time % 1440):
                day_offset += 1440

            current_time = departure + day_offset

    end_time = current_time

    return end_time - start_time

In [ ]:
journey_duration = (
    clean_df.groupby("Train_No")
    .apply(calculate_journey_duration, include_groups=False)
    .reset_index(name="Journey_Duration_Minutes")
)

NameError: name 'clean_df' is not defined

In [15]:
journey_duration.head()

,Train_No,Journey_Duration_Minutes
0,107,105.0
1,108,115.0
2,128,1325.0
3,290,9120.0
4,401,2190.0


In [16]:
journey_duration["Journey_Duration_Minutes"].describe()

count    11113.000000
mean       433.969045
std        669.147641
min          5.000000
25%         63.000000
50%        135.000000
75%        450.000000
max       9120.000000
Name: Journey_Duration_Minutes, dtype: float64

In [17]:
total_distance= (
    clean_df.groupby("Train_No")["Distance"]
    .max()
    .reset_index(name="Total_Distance")
)

In [18]:
total_stops=(
    clean_df.groupby("Train_No")
    .size()
    .reset_index(name="Number_of_Stops")
)

In [19]:
ml_df= journey_duration.merge(
    total_distance,
    on="Train_No"
)

ml_df=ml_df.merge(
    total_stops,
    on="Train_No"
)

In [20]:
ml_df.head()

,Train_No,Journey_Duration_Minutes,Total_Distance,Number_of_Stops
0,107,105.0,78,4
1,108,115.0,83,4
2,128,1325.0,978,22
3,290,9120.0,2694,14
4,401,2190.0,1618,12


In [21]:
ml_df.shape

(11113, 4)

In [24]:
ml_df.to_csv("Dataset/ML_Train_Level_Data.csv", index=False)